# Import Library

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Variable
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
#import optuna
import pickle
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
#import seaborn as sns
from matplotlib import font_manager, rc
import platform
import os
import pandas as pd
import numpy as np
from pykalman import KalmanFilter
import matplotlib.dates as mdates
pd.set_option('display.max_columns', None)
#import seaborn as sns
#import plotly.graph_objects as go
import glob
from sklearn.metrics import r2_score, mean_squared_error

# Set the font properties
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트 사용시 마이너스 기호 깨짐 방지

#### Set seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available)
print(torch.version.cuda)
print(device)

class GLU(nn.Module):
 
    def __init__(self, input_size):
        super().__init__()
        
        # Input
        self.a = nn.Linear(input_size, input_size)

        # Gate
        self.sigmoid = nn.Sigmoid()
        self.b = nn.Linear(input_size, input_size)
        
    def forward(self, x):
        """
        Args:
            x (torch.tensor): tensor passing through the gate
        """
        gate = self.sigmoid(self.b(x))
        x = self.a(x)
        
        return torch.mul(gate, x)

class TemporalLayer(nn.Module):
    def __init__(self, module):
        super().__init__()
        self.module = module


    def forward(self, x):
    
        t, n = x.size(0), x.size(1)
        x = x.reshape(t * n, -1)
        x = self.module(x)
        x = x.reshape(t, n, x.size(-1))

        return x

class GatedResidualNetwork(nn.Module):
    
    def __init__(self, input_size, hidden_size, output_size, dropout, context_size=None, is_temporal=True):
        super().__init__()

        self.input_size = input_size
        self.output_size = output_size
        self.context_size = context_size
        self.hidden_size = hidden_size
        self.dropout = dropout
        self.is_temporal = is_temporal
        
        if self.is_temporal:
            if self.input_size != self.output_size:
                self.skip_layer = TemporalLayer(nn.Linear(self.input_size, self.output_size))

            # Context vector c
            if self.context_size != None:
                self.c = TemporalLayer(nn.Linear(self.context_size, self.hidden_size, bias=False))

            # Dense & ELU
            self.dense1 = TemporalLayer(nn.Linear(self.input_size, self.hidden_size))
            self.elu = nn.ELU()

            # Dense & Dropout
            self.dense2 = TemporalLayer(nn.Linear(self.hidden_size,  self.output_size))
            self.dropout = nn.Dropout(self.dropout)

            # Gate, Add & Norm
            self.gate = TemporalLayer(GLU(self.output_size))
            self.layer_norm = TemporalLayer(nn.BatchNorm1d(self.output_size))

        else:
            if self.input_size != self.output_size:
                self.skip_layer = nn.Linear(self.input_size, self.output_size)

            # Context vector c
            if self.context_size != None:
                self.c = nn.Linear(self.context_size, self.hidden_size, bias=False)

            # Dense & ELU
            self.dense1 = nn.Linear(self.input_size, self.hidden_size)
            self.elu = nn.ELU()

            # Dense & Dropout
            self.dense2 = nn.Linear(self.hidden_size,  self.output_size)
            self.dropout = nn.Dropout(self.dropout)

            # Gate, Add & Norm
            self.gate = GLU(self.output_size)
            self.layer_norm = nn.BatchNorm1d(self.output_size)


    def forward(self, x, c=None):

        if self.input_size!=self.output_size:
            a = self.skip_layer(x)
        else:
            a = x
        
        x = self.dense1(x)

        if c != None:
            c = self.c(c.unsqueeze(1))
            x += c

        eta_2 = self.elu(x)
        
        eta_1 = self.dense2(eta_2)
        eta_1 = self.dropout(eta_1)

        gate = self.gate(eta_1)
        gate += a
        if gate.size(0) > 1:  # Only apply BatchNorm if batch size is greater than 1
            x = self.layer_norm(gate)
        else:
            x = gate  # Skip normalization if batch size is too small
        
        return x

class VariableSelectionNetwork(nn.Module):
  
    def __init__(self, input_size, output_size, hidden_size, dropout, context_size=None, is_temporal=True):
        super().__init__()

        self.hidden_size = hidden_size
        self.input_size = input_size
        self.output_size = output_size
        self.dropout = dropout
        self.context_size = context_size
        self.is_temporal = is_temporal
       
        self.flattened_inputs = GatedResidualNetwork(self.output_size*self.input_size, 
                                                     self.hidden_size, self.output_size, 
                                                     self.dropout, self.context_size, 
                                                     self.is_temporal)
        
        self.transformed_inputs = nn.ModuleList(
            [GatedResidualNetwork(
                self.input_size, self.hidden_size, self.hidden_size, 
                self.dropout, self.context_size, self.is_temporal) for i in range(self.output_size)])

        self.softmax = nn.Softmax(dim=-1)


    def forward(self, embedding, context=None):

        # Generation of variable selection weights
        sparse_weights = self.flattened_inputs(embedding, context)
        if self.is_temporal:
            sparse_weights = self.softmax(sparse_weights).unsqueeze(2)
        else:
            sparse_weights = self.softmax(sparse_weights).unsqueeze(1)

        # Additional non-linear processing for each feature vector
        transformed_embeddings = torch.stack(
            [self.transformed_inputs[i](embedding[
                Ellipsis, i*self.input_size:(i+1)*self.input_size]) for i in range(self.output_size)], axis=-1)

        # Processed features are weighted by their corresponding weights and combined
        combined = transformed_embeddings*sparse_weights
        combined = combined.sum(axis=-1)

        return combined, sparse_weights

class ScaledDotProductAttention(nn.Module):    #MUTLI-HEAD SELF-ATTENTION
  
    def __init__(self, dropout=0.0): 
        super().__init__()

        self.dropout = nn.Dropout(dropout)
        self.softmax = nn.Softmax(dim=2)
        

    def forward(self, query, key, value, mask=None):

        d_k = key.shape[-1]
        scaling_factor = torch.sqrt(torch.tensor(d_k).to(torch.float32))

        scaled_dot_product = torch.matmul(query, key.permute(0,2,1)) / scaling_factor 
        if mask != None:
            scaled_dot_product = scaled_dot_product.masked_fill(mask == 0, -1e9)
        attention = self.softmax(scaled_dot_product)
        attention = self.dropout(attention)
        output = torch.matmul(attention, value)

        return output, attention

class InterpretableMultiHeadAttention(nn.Module):
   
    def __init__(self, num_attention_heads, hidden_size, dropout=0.0):
        super().__init__()

        self.num_attention_heads = num_attention_heads
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout)

        self.qs = nn.ModuleList([nn.Linear(self.hidden_size, self.hidden_size, bias=False) for i in range(self.num_attention_heads)])
        self.ks = nn.ModuleList([nn.Linear(self.hidden_size, self.hidden_size, bias=False) for i in range(self.num_attention_heads)])

        vs_layer = nn.Linear(self.hidden_size, self.hidden_size, bias=False) # Value is shared for improved interpretability
        self.vs = nn.ModuleList([vs_layer for i in range(self.num_attention_heads)])

        self.attention = ScaledDotProductAttention()
        self.linear = nn.Linear(self.hidden_size, self.hidden_size, bias=False)


    def forward(self, query, key, value, mask=None):
        
        batch_size, tgt_len, embed_dim = query.shape
        head_dim = embed_dim // self.num_attention_heads

        # Now we iterate over each head to calculate outputs and attention
        heads = []
        attentions = []

        for i in range(self.num_attention_heads):
            q_i = self.qs[i](query)
            k_i = self.ks[i](key)
            v_i = self.vs[i](value)

            # Reshape q, k, v for multihead attention
            q_i = query.reshape(batch_size, tgt_len, self.num_attention_heads, head_dim).transpose(1,2).reshape(batch_size*self.num_attention_heads, tgt_len, head_dim)
            k_i = key.reshape(batch_size, tgt_len, self.num_attention_heads, head_dim).transpose(1,2).reshape(batch_size*self.num_attention_heads, tgt_len, head_dim)
            v_i = value.reshape(batch_size, tgt_len, self.num_attention_heads, head_dim).transpose(1,2).reshape(batch_size*self.num_attention_heads, tgt_len, head_dim)

            head, attention = self.attention(q_i, k_i, v_i, mask)

            # Revert to original target shape
            head = head.reshape(batch_size, self.num_attention_heads, tgt_len, head_dim).transpose(1,2).reshape(-1, tgt_len, self.num_attention_heads*head_dim)
            head_dropout = self.dropout(head)
            heads.append(head_dropout)
            attentions.append(attention)

        # Output the results
        if self.num_attention_heads > 1:
            heads = torch.stack(heads, dim=2) #.reshape(batch_size, tgt_len, -1, self.hidden_size)
            outputs = torch.mean(heads, dim=2)
        else:
            outputs = head

        attentions = torch.stack(attentions, dim=2)
        attention = torch.mean(attentions, dim=2)
        
        outputs = self.linear(outputs)
        outputs = self.dropout(outputs)

        return outputs, attention

class QuantileLoss(nn.Module):

    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = quantiles
        
    def forward(self, preds, target):
    
        assert not target.requires_grad
        assert preds.size(0) == target.size(0)

        losses = []
        for i, q in enumerate(self.quantiles):  
            errors = target - preds[:, :, i]
            losses.append(torch.max((q-1) * errors, q * errors).unsqueeze(1))
        
        loss = torch.mean(torch.sum(torch.cat(losses, dim=2), dim=2))

        return loss

class CustomMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, preds, targets):
        """
        preds:   (batch, time, quantiles) 예측값
        targets: (batch, time)            실제값

        자동으로 targets shape을 (batch, time, quantiles)로 확장하여 계산
        """
        assert preds.size(0) == targets.size(0)
        assert preds.size(1) == targets.size(1)

        # targets: (batch, time) → (batch, time, 1)
        if targets.ndim == 2:
            targets = targets.unsqueeze(2)

        # (batch, time, 1) → (batch, time, num_quantiles)
        targets = targets.expand_as(preds)

        # MSE 계산
        loss = torch.mean((preds - targets) ** 2)
        return loss

class TFT(nn.Module):
  
    def __init__(
        self, 
        quantiles, 
        dropout,
        device,
        hidden_layer_size,
        num_lstm_layers,
        embedding_dim,
        encoder_steps,
        num_attention_heads,
        col_to_idx,
        static_covariates,
        time_dependent_categorical,
        time_dependent_continuous,
        category_counts,
        known_time_dependent,
        observed_time_dependent,
    ):

        super().__init__()

        # Inputs
        self.col_to_idx = col_to_idx
        self.static_covariates = static_covariates
        self.time_dependent_categorical = time_dependent_categorical
        self.time_dependent_continuous = time_dependent_continuous
        self.category_counts = category_counts
        self.known_time_dependent = known_time_dependent
        self.observed_time_dependent = observed_time_dependent
        self.time_dependent = self.known_time_dependent+self.observed_time_dependent

        # Architecture
        self.encoder_steps = encoder_steps
        self.hidden_size = hidden_layer_size
        self.num_lstm_layers = num_lstm_layers
        self.dropout = dropout
        self.embedding_dim = embedding_dim
        self.num_attention_heads = num_attention_heads

        # Outputs
        self.quantiles = quantiles

        # Other
        self.device = device

        # Prepare embeddings for the static covariates and static context vectors
        self.static_embeddings = nn.ModuleDict({col: nn.Embedding(self.category_counts[col], self.embedding_dim).to(self.device) for col in self.static_covariates}) 
        self.static_variable_selection = VariableSelectionNetwork(self.embedding_dim, len(self.static_covariates), self.hidden_size, self.dropout, is_temporal=False) 

        self.static_context_variable_selection = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout, is_temporal=False)
        self.static_context_enrichment = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout, is_temporal=False)
        self.static_context_state_h = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout, is_temporal=False)
        self.static_context_state_c = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout, is_temporal=False)
        
        # Prepare embeddings and linear transformations for time dependent variables
        self.temporal_cat_embeddings = nn.ModuleDict({col: TemporalLayer(nn.Embedding(self.category_counts[col], self.embedding_dim)).to(self.device) for col in self.time_dependent_categorical})
        self.temporal_real_transformations = nn.ModuleDict({col: TemporalLayer(nn.Linear(1, self.embedding_dim)).to(self.device) for col in self.time_dependent_continuous})

        # Variable selection and encoder for past inputs
        self.past_variable_selection = VariableSelectionNetwork(self.embedding_dim, len(self.time_dependent), self.hidden_size, self.dropout, context_size=self.hidden_size)

        # Variable selection and decoder for known future inputs
        self.future_variable_selection = VariableSelectionNetwork(self.embedding_dim, len([col for col in self.time_dependent if col not in self.observed_time_dependent]), 
                                                                  self.hidden_size, self.dropout, context_size=self.hidden_size)

        # LSTM encoder and decoder
        self.lstm_encoder = nn.LSTM(input_size=self.hidden_size, hidden_size=self.hidden_size, num_layers=self.num_lstm_layers, dropout=self.dropout)
        self.lstm_decoder = nn.LSTM(input_size=self.hidden_size, hidden_size=self.hidden_size, num_layers=self.num_lstm_layers, dropout=self.dropout)

        # Gated skip connection and normalization
        self.gated_skip_connection = TemporalLayer(GLU(self.hidden_size))
        self.add_norm = TemporalLayer(nn.BatchNorm1d(self.hidden_size))

        # Temporal Fusion Decoder

        # Static enrichment layer
        self.static_enrichment = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout, self.hidden_size)
        
        # Temporal Self-attention layer
        self.multihead_attn = InterpretableMultiHeadAttention(self.num_attention_heads, self.hidden_size)
        self.attention_gated_skip_connection = TemporalLayer(GLU(self.hidden_size))
        self.attention_add_norm = TemporalLayer(nn.BatchNorm1d(self.hidden_size, self.hidden_size))

        # Position-wise feed-forward layer
        self.position_wise_feed_forward = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout)

        # Output layer
        self.output_gated_skip_connection = TemporalLayer(GLU(self.hidden_size))
        self.output_add_norm = TemporalLayer(nn.BatchNorm1d(self.hidden_size, self.hidden_size))

        self.output = TemporalLayer(nn.Linear(self.hidden_size, len(self.quantiles)))
        
  
    # def define_static_covariate_encoders(self, x):
    #     embedding_vectors = [self.static_embeddings[col](x[:, 0, self.col_to_idx[col]].long().to(self.device)) for col in self.static_covariates]
    #     static_embedding = torch.cat(embedding_vectors, dim=1)
    #     static_encoder, static_weights = self.static_variable_selection(static_embedding)

    #     # Static context vectors
    #     static_context_s = self.static_context_variable_selection(static_encoder) # Context for temporal variable selection
    #     static_context_e = self.static_context_enrichment(static_encoder) # Context for static enrichment layer
    #     static_context_h = self.static_context_state_h(static_encoder) # Context for local processing of temporal features (encoder/decoder)
    #     static_context_c = self.static_context_state_c(static_encoder) # Context for local processing of temporal features (encoder/decoder)

    #     return static_encoder, static_weights, static_context_s, static_context_e, static_context_h, static_context_c

    def define_static_covariate_encoders(self, x):
        if len(self.static_covariates) == 0:
            batch_size = x.size(0)
            dummy = torch.zeros(batch_size, self.hidden_size).to(self.device)
            return dummy, None, dummy, dummy, dummy, dummy

        embedding_vectors = [self.static_embeddings[col](x[:, 0, self.col_to_idx[col]].long().to(self.device)) for col in self.static_covariates]
        static_embedding = torch.cat(embedding_vectors, dim=1)
        static_encoder, static_weights = self.static_variable_selection(static_embedding)

        # Static context vectors
        static_context_s = self.static_context_variable_selection(static_encoder)
        static_context_e = self.static_context_enrichment(static_encoder)
        static_context_h = self.static_context_state_h(static_encoder)
        static_context_c = self.static_context_state_c(static_encoder)

        return static_encoder, static_weights, static_context_s, static_context_e, static_context_h, static_context_c


    def define_past_inputs_encoder(self, x, context):
        # print("Time dependent categorical variables (past):", self.time_dependent_categorical)
        # print("Time dependent continuous variables (past):", self.time_dependent_continuous)

        # Only concatenate embeddings if there are any time-dependent categorical variables
        if self.time_dependent_categorical:
            embedding_vectors = torch.cat([self.temporal_cat_embeddings[col](x[:, :, self.col_to_idx[col]].long()) for col in self.time_dependent_categorical], dim=2)
        else:
            embedding_vectors = torch.empty((x.size(0), x.size(1), 0)).to(self.device)

        transformation_vectors = torch.cat([self.temporal_real_transformations[col](x[:, :, self.col_to_idx[col]]) for col in self.time_dependent_continuous], dim=2)

        past_inputs = torch.cat([embedding_vectors, transformation_vectors], dim=2)
        past_encoder, past_weights = self.past_variable_selection(past_inputs, context)

        return past_encoder.transpose(0, 1), past_weights


    def define_known_future_inputs_decoder(self, x, context):
        # print("Time dependent continuous variables:", self.time_dependent_continuous)
        # print("Observed time dependent variables:", self.observed_time_dependent)

        # Only concatenate embeddings if there are any time-dependent categorical variables
        if self.time_dependent_categorical:
            embedding_vectors = torch.cat([self.temporal_cat_embeddings[col](x[:, :, self.col_to_idx[col]].long()) 
                                        for col in self.time_dependent_categorical if col not in self.observed_time_dependent], dim=2)
        else:
            embedding_vectors = torch.empty((x.size(0), x.size(1), 0)).to(self.device)

        transformation_vectors = torch.cat([self.temporal_real_transformations[col](x[:, :, self.col_to_idx[col]]) 
                                            for col in self.time_dependent_continuous if col not in self.observed_time_dependent], dim=2)

        future_inputs = torch.cat([embedding_vectors, transformation_vectors], dim=2)
        future_decoder, future_weights = self.future_variable_selection(future_inputs, context)

        return future_decoder.transpose(0, 1), future_weights

    def define_lstm_encoder(self, x, static_context_h, static_context_c):
        output, (state_h, state_c) = self.lstm_encoder(x, (static_context_h.unsqueeze(0).repeat(self.num_lstm_layers,1,1), 
                                                           static_context_c.unsqueeze(0).repeat(self.num_lstm_layers,1,1)))
        
        return output, state_h, state_c


    def define_lstm_decoder(self, x, state_h, state_c):
        output, (_, _) = self.lstm_decoder(x, (state_h.unsqueeze(0).repeat(self.num_lstm_layers,1,1), 
                                               state_c.unsqueeze(0).repeat(self.num_lstm_layers,1,1)))
        
        return output

    
    def get_mask(self, attention_inputs):
        #mask = torch.cumsum(torch.eye(attention_inputs.shape[1]*self.num_attention_heads, attention_inputs.shape[0]), dim=1)
        mask = torch.cumsum(torch.eye(attention_inputs.shape[0]*self.num_attention_heads, attention_inputs.shape[1]), dim=1)

        return mask.unsqueeze(2).to(self.device)

    def forward(self, x):

        # Static variable selection and static covariate encoders
        static_encoder, static_weights, static_context_s, static_context_e, static_context_h, static_context_c = self.define_static_covariate_encoders(x)

        # Past input variable selection and LSTM encoder
        past_encoder, past_weights = self.define_past_inputs_encoder(x[:, :self.encoder_steps, :].float().to(self.device), static_context_s)

        # Known future inputs variable selection and LSTM decoder
        future_decoder, future_weights = self.define_known_future_inputs_decoder(x[:, self.encoder_steps:, :].float().to(self.device), static_context_s)

        # Pass output from variable selection through LSTM encoder and decoder
        encoder_output, state_h, state_c = self.define_lstm_encoder(past_encoder, static_context_h, static_context_c)
        decoder_output = self.define_lstm_decoder(future_decoder, static_context_h, static_context_c)

        # Gated skip connection before moving into the Temporal Fusion Decoder
        variable_selection_outputs = torch.cat([past_encoder, future_decoder], dim=0)
        lstm_outputs = torch.cat([encoder_output, decoder_output], dim=0)
        gated_outputs = self.gated_skip_connection(lstm_outputs)
        temporal_feature_outputs = self.add_norm(variable_selection_outputs.add(gated_outputs))
        temporal_feature_outputs = temporal_feature_outputs.transpose(0, 1)

        # Temporal Fusion Decoder
        # Static enrcihment layer
        static_enrichment_outputs = self.static_enrichment(temporal_feature_outputs, static_context_e)

        # Temporal Self-attention layer
        mask = self.get_mask(static_enrichment_outputs)
        multihead_outputs, multihead_attention = self.multihead_attn(static_enrichment_outputs, static_enrichment_outputs, static_enrichment_outputs, mask=mask)
        
        attention_gated_outputs = self.attention_gated_skip_connection(multihead_outputs)
        attention_outputs = self.attention_add_norm(attention_gated_outputs.add(static_enrichment_outputs))

        # Position-wise feed-forward layer
        temporal_fusion_decoder_outputs = self.position_wise_feed_forward(attention_outputs)

        # Output layer
        gate_outputs = self.output_gated_skip_connection(temporal_fusion_decoder_outputs)
        norm_outputs = self.output_add_norm(gate_outputs.add(temporal_feature_outputs))

        output = self.output(norm_outputs[:, self.encoder_steps:, :])
        
        attention_weights = {
            'multihead_attention': multihead_attention,
            'static_weights': static_weights[..., 0] if static_weights is not None else None,
            'past_weights': past_weights[..., 0, :] if past_weights is not None else None,
            'future_weights': future_weights[..., 0, :] if future_weights is not None else None
        }
        return  output
    
    
import os
import glob
import torch
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

# Parameter

In [ ]:
# === Directory ===
os.chdir('/home/uoswem7/SaaS/논문용_모델/WTemp')

# === HYPERPARAMETERS ===
quantiles = [0.2, 0.5, 0.8]
Input_Sequence = 36
Output_Sequence = 7
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
targets = ['PD1.WTemp', 'PD2.WTemp', 'PD3.WTemp']

Dropout_list = [0.0]
lr_list = [0.005, 0.0001, 0.0005]
Num_LSTM_Layers_list = [3, 5, 7]
Hidden_Layer_Size_list = [16, 32]
batch_size_list = [64, 128]
Embedding_Dim_list = [8]
Num_Attention_list = [4]

EPOCHS = 100
pickle_dir = "./Model/WTemp_TFT/mse_loss"
os.makedirs(pickle_dir, exist_ok=True)

# Modeling

In [ ]:
for target in targets:
    filepath = '/home/uoswem7/SaaS/논문용_모델/WTemp/Model/WTemp_TFT/TFT_data_%s.csv'%(target)
    # filepath = '/home/uoswem4/TFT_TL_NEW/TFT_data_PD2.WTemp.csv'

    # === LOAD DATA ===
    df = pd.read_csv(filepath, encoding='cp949')
    # Define feature ordering
    input_cols = [f'var_{i}' for i in range(14)] + ['known_var']
    col_to_idx = {col: i for i, col in enumerate(input_cols)}
    # === BUILD SEQUENCES ===
    def create_sequences_from_stream_debug(df, input_len=36, output_len=7, print_every=1):
        obs_vars = [f'var_{i}' for i in range(14)]
        known_vars = ['known_var']
        target_var = ['target']

        df_sorted = df.sort_values(['sample_id', 'time_idx']).reset_index(drop=True)

        X_seq_list = []
        y_seq_list = []

        total_len = input_len + output_len
        debug_counter = 0

        for i in range(len(df_sorted) - total_len + 1):
            window = df_sorted.iloc[i:i + total_len]

            if not np.all(window['sample_id'].values == window['sample_id'].iloc[0]):
                continue

            x_obs = window.iloc[0:input_len][obs_vars].values
            x_known = window.iloc[0:total_len][known_vars].values
            decoder_known = x_known[input_len:]

            if decoder_known.shape[0] != output_len:
                continue

            x_encoder = np.concatenate([x_obs, x_known[:input_len]], axis=1)
            zero_obs = np.zeros((output_len, len(obs_vars)))
            decoder_input = np.concatenate([zero_obs, decoder_known], axis=1)
            full_input = np.concatenate([x_encoder, decoder_input], axis=0)

            y_target = window.iloc[input_len:][target_var].values.squeeze()

            X_seq_list.append(full_input)
            y_seq_list.append(y_target)

            # # === Debug Output ===
            # if debug_counter % print_every == 0:
            #     print(f"\n=== Sample {debug_counter} (from rows {i}-{i+total_len-1}) ===")
            #     print(f"sample_id: {window['sample_id'].iloc[0]}")
            #     print(f"time_idx range: {window['time_idx'].min()} to {window['time_idx'].max()}")
            #     print("Input sequence shape:", full_input.shape)
            #     print("Target sequence shape:", y_target.shape)

            #     print("\n--- Encoder (First 3 rows) ---")
            #     print(pd.DataFrame(full_input[:3], columns=[*obs_vars, *known_vars]))

            #     print("\n--- Decoder (Last 3 rows) ---")
            #     print(pd.DataFrame(full_input[-3:], columns=[*obs_vars, *known_vars]))

            #     print("\n--- Target values ---")
            #     print(y_target)

            # debug_counter += 1

        return np.stack(X_seq_list), np.stack(y_seq_list)


    XX_train, yy_train = create_sequences_from_stream_debug(df, input_len=36, output_len=7, print_every=500)

    print(f"Training data shape: {XX_train.shape}, Target shape: {yy_train.shape}")

    # === DATASET ===
    class TimeSeriesDataset(Dataset):
        def __init__(self, x, y, device):
            self.x = torch.tensor(x, dtype=torch.float32).to(device)
            self.y = torch.tensor(y, dtype=torch.float32).to(device)

        def __len__(self):
            return len(self.x)

        def __getitem__(self, index):
            return self.x[index], self.y[index]

    # === TRAINING FUNCTION ===
    def train_model(model, train_dataloader, criterion, optimizer, scheduler, EPOCHS, random_seed=42):
        torch.manual_seed(random_seed)
        torch.cuda.manual_seed(random_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        last_lr = optimizer.param_groups[0]['lr']

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0
            count = 0

            for inputs, targets in train_dataloader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                count += 1

            avg_loss = total_loss / count if count > 0 else float('inf')

            current_lr = optimizer.param_groups[0]['lr']
            if epoch % 5 == 0 or current_lr != last_lr:
                print(f"Epoch {epoch}/{EPOCHS}, Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")
            if current_lr != last_lr:
                print(f"Learning rate changed to {current_lr:.6f} at epoch {epoch}")
                last_lr = current_lr

            scheduler.step(avg_loss)

        return model

    # === HYPERPARAMETERS ===
    Dropout_list = [0.0]
    lr_list = [0.005, 0.0001, 0.0005]
    Num_LSTM_Layers_list = [3, 5, 7]
    Hidden_Layer_Size_list = [16, 32]
    batch_size_list = [64, 128]
    Embedding_Dim_list = [8]
    Num_Attention_list = [4]

    EPOCHS = 100

    pickle_dir = "/home/uoswem7/SaaS/논문용_모델/WTemp/Model/WTemp_TFT/mse_loss"
    os.makedirs(pickle_dir, exist_ok=True)

    # === TRAINING LOOP ===
    for Dropout in Dropout_list:
        for learning_rate in lr_list:
            for Num_LSTM_Layers in Num_LSTM_Layers_list:
                for Hidden_Layer_Size in Hidden_Layer_Size_list:
                    for Embedding_Dim in Embedding_Dim_list:
                        for Num_Attention_Heads in Num_Attention_list:
                            for batch_size in batch_size_list:
                                if Hidden_Layer_Size % Num_Attention_Heads != 0:
                                    print(f"Skipping incompatible config: Hidden_Layer_Size={Hidden_Layer_Size}, Num_Attention_Heads={Num_Attention_Heads}")
                                    continue


                                model_save_path = os.path.join(
                                    pickle_dir,
                                    f"{target}_dr{Dropout}_lr{learning_rate}_nl{Num_LSTM_Layers}_hid{Hidden_Layer_Size}_emb{Embedding_Dim}_atten{Num_Attention_Heads}_bs{batch_size}.pt"
                                )

                                print(f"\n>> Training Config:")
                                print(f"Dropout: {Dropout}, LR: {learning_rate}, LSTM Layers: {Num_LSTM_Layers}, Hidden: {Hidden_Layer_Size}, Embedding: {Embedding_Dim}, Attention: {Num_Attention_Heads}, Batch Size: {batch_size}")

                                # Define your TFT model here — this assumes you already have a TFT class defined
                                model = TFT(
                                    quantiles=quantiles,
                                    dropout=Dropout,
                                    device=device,
                                    hidden_layer_size=Hidden_Layer_Size,
                                    num_lstm_layers=Num_LSTM_Layers,
                                    embedding_dim=Embedding_Dim,
                                    encoder_steps=Input_Sequence,
                                    num_attention_heads=Num_Attention_Heads,
                                    col_to_idx=col_to_idx,  # or leave this out if not needed
                                    static_covariates=[],
                                    time_dependent_categorical=[],
                                    time_dependent_continuous=[f'var_{i}' for i in range(14)] + ['known_var'],
                                    category_counts={},
                                    known_time_dependent=['known_var'],
                                    observed_time_dependent=[f'var_{i}' for i in range(14)],
                                ).to(device)

                                optimizer = optim.Adam(model.parameters(), lr=learning_rate)
                                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=15)
                                # criterion = QuantileLoss(quantiles=quantiles)
                                criterion = CustomMSELoss()

                                train_dataset = TimeSeriesDataset(XX_train, yy_train, device)
                                train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

                                model = train_model(model, train_dataloader, criterion, optimizer, scheduler, EPOCHS)
                                model.eval()
                                model.to("cpu")

                                torch.save(model, model_save_path)
                                print(f"✅ Model saved to: {model_save_path}")

                                del model